# Demostrador del pipeline de restauración

TFM · Restauración y super-resolución de imágenes históricas mediante GAN

---

Interfaz que permite cargar una fotografía, marcar la zona dañada, ejecutar el pipeline
completo (inpainting con LaMa seguido de super-resolución con A-ESRGAN) y comparar el
resultado con el original.

**El diseño no es arbitrario.** Tres de las cuatro medidas de mitigación propuestas en la
sección de consideraciones éticas se traducen directamente en funcionalidad:

| Medida de la sección de ética | Cómo se materializa |
|---|---|
| Conservar la imagen degradada junto a la restaurada | Vista comparativa con deslizador; nunca se muestra la salida sola |
| Tratar con cautela las máscaras sobre rasgos faciales | Aviso automático cuando la región marcada se solapa con un rostro |
| No añadir coloración ni estilización | Decisión de arquitectura: el pipeline solo repara e incrementa resolución |
| Documentar la salida como versión procesada mediante IA | Pie identificativo en la imagen descargada |

**Dónde se ejecuta.** Este notebook levanta la interfaz sobre la GPU de la sesión de
Colab y devuelve un enlace público temporal. Sirve para desarrollo y como respaldo. El
despliegue permanente se hará después en un Space de Hugging Face, con el mismo código.

## 1. Entorno

In [ ]:
%pip install -q gradio "basicsr>=1.3.3.11" simple-lama-inpainting

# Pin de versiones críticas para evitar el bug de numpy._core durante la sesión.
# Cuando varios pip install ocurren en la misma sesión (basicsr, simple-lama-inpainting...)
# pueden mezclar archivos de distintas versiones de numpy y romper imports con:
#     ImportError: cannot import name '_center' from 'numpy._core.umath'
# o con ValueError: numpy.dtype size changed, may indicate binary incompatibility.
# Se fuerza aquí un conjunto internamente consistente con --force-reinstall y se deja
# un PIP_CONSTRAINT para que las instalaciones posteriores no vuelvan a degradarlo.
# (Mismo criterio que en LaMa_FineTune_clean, celda 2.)
import subprocess, sys, os
from pathlib import Path

PINS = [
    'numpy==2.0.2',           # último parche estable de la rama 2.0
    'scipy==1.14.1',          # exige numpy >= 2.0, < 2.2
    'scikit-image==0.24.0',   # compatible con numpy 2.0
]
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', *PINS],
    check=True,
)

_CONSTRAINT = Path('/content/numpy_pins.txt')
_CONSTRAINT.write_text('\n'.join(PINS))
os.environ['PIP_CONSTRAINT'] = str(_CONSTRAINT)
print('Pin aplicado:', PINS)

import site
from pathlib import Path

# torchvision >= 0.16 eliminó functional_tensor, que basicsr todavía importa.
for base in site.getsitepackages() + [site.getusersitepackages()]:
    p = Path(base) / 'basicsr' / 'data' / 'degradations.py'
    if p.exists():
        t = p.read_text()
        viejo = 'from torchvision.transforms.functional_tensor import rgb_to_grayscale'
        nuevo = 'from torchvision.transforms.functional import rgb_to_grayscale'
        if viejo in t:
            p.write_text(t.replace(viejo, nuevo)); print('Parche basicsr aplicado')

import PIL._typing
if not hasattr(PIL._typing, '_Ink'):
    from typing import Union
    PIL._typing._Ink = Union[int, tuple]

import gradio, torch
print('gradio', gradio.__version__, '| torch', torch.__version__,
      '| GPU', torch.cuda.is_available())

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, sys, io, time, subprocess, contextlib
from pathlib import Path

import numpy as np
import cv2
import torch
import gradio as gr
from PIL import Image, ImageDraw, ImageFont

PROJECT_DIR = Path('/content/drive/MyDrive/TFM')
AESRGAN_DIR = Path('/content/A-ESRGAN')
LAMA_DIR    = Path('/content/lama')          # <- ajustar a tu instalación

PESOS = {
    'preentrenado': AESRGAN_DIR/'experiments'/'pretrained_models'/'A_ESRGAN_Single.pth',
    'ajustado':     PROJECT_DIR/'Fase4c'/'checkpoints'/'brazoF_flick'/'models'/'net_g_400.pth',
}

ESCALA   = 4
LADO_MAX = 1024      # cota del lado mayor de la ENTRADA; a 4x la salida llega a 4096
TILE     = 400

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)
if DEVICE == 'cpu':
    print('AVISO: en CPU cada imagen tarda decenas de segundos. Activar GPU en Colab.')

# Repositorio y pesos de A-ESRGAN
REPO_URL  = 'https://github.com/stroking-fishes-ml-corp/A-ESRGAN.git'
MODEL_URL = ('https://github.com/stroking-fishes-ml-corp/A-ESRGAN/releases/'
             'download/v1.0.0/A_ESRGAN_Single.pth')
if not AESRGAN_DIR.exists():
    subprocess.run(['git','clone',REPO_URL,str(AESRGAN_DIR)], check=True)
PESOS['preentrenado'].parent.mkdir(parents=True, exist_ok=True)
if not PESOS['preentrenado'].exists():
    p = subprocess.run(['wget','-O',str(PESOS['preentrenado']),MODEL_URL],
                       capture_output=True, text=True)
    if p.returncode:
        PESOS['preentrenado'].unlink(missing_ok=True)
        print(p.stderr); raise RuntimeError('Descarga fallida')

sys.path.insert(0, str(AESRGAN_DIR))
import importlib; importlib.invalidate_caches()

for k, v in PESOS.items():
    print(f'{k:13s}: {"OK   " if v.exists() else "FALTA"} {v}')

## 2. Modelos

A-ESRGAN se carga una vez por juego de pesos y se mantiene en memoria. LaMa se conecta
mediante una función adaptadora con un contrato explícito, igual que se hizo en las fases
de evaluación: **recibe la imagen y la máscara, devuelve la imagen reparada, ambas del
mismo tamaño**. Rellenar esa función es lo único que hay que adaptar a tu instalación.

Si LaMa no está disponible, la interfaz sigue funcionando en modo solo super-resolución y
lo indica, en lugar de fallar.

In [ ]:
def cargar_upsampler(model_path, tile=TILE):
    from aesrgan.utils import RealESRGANer
    from basicsr.archs.rrdbnet_arch import RRDBNet
    red = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64,
                  num_block=23, num_grow_ch=32, scale=4)
    return RealESRGANer(scale=4, model_path=str(model_path), model=red,
                        tile=tile, tile_pad=10, pre_pad=0, half=False)

UPS = {}
for k, v in PESOS.items():
    if v.exists():
        UPS[k] = cargar_upsampler(v)
        print(f'{k}: cargado en {next(UPS[k].model.parameters()).device}')

In [ ]:
# ---------------------------------------------------------------------------
# ADAPTADOR DE LaMa
#
# Contrato:
#   entrada  img_bgr  uint8 HxWx3   |  mask uint8 HxW, 255 = reparar
#   salida            uint8 HxWx3, mismo tamaño que la entrada
#
# Sustituir el cuerpo por la llamada a tu instalación de LaMa.
# ---------------------------------------------------------------------------
# La instalación real usa el paquete simple-lama-inpainting (SimpleLama), que
# trabaja con objetos PIL y ya resuelve internamente el relleno a múltiplo de 8,
# la normalización y el umbral de la máscara. La convención de máscara -255 =
# reparar- coincide con la del contrato de esta función, sin traducción alguna.
LAMA_CKPT = PROJECT_DIR / 'checkpoints' / 'lama_finetuned' / 'lama_best.pth'

LAMA_DISPONIBLE = False
_LAMA = None

def cargar_lama():
    from simple_lama_inpainting import SimpleLama
    lama = SimpleLama()                       # pesos preentrenados big-lama, en caché
    if LAMA_CKPT.exists():
        estado = torch.load(LAMA_CKPT, map_location=DEVICE)
        lama.model.load_state_dict(estado, strict=False)
        print(f'Pesos ajustados cargados desde {LAMA_CKPT.name}')
    else:
        print(f'{LAMA_CKPT} no encontrado; se usa el modelo preentrenado de LaMa')
    lama.model.to(DEVICE).eval()
    return lama


def inpaint_lama(img_bgr, mask):
    """img_bgr: uint8 HxWx3 BGR | mask: uint8 HxW, 255 = reparar.
    SimpleLama espera PIL: la imagen en RGB y la máscara en modo 'L'."""
    img_pil  = Image.fromarray(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    mask_pil = Image.fromarray(mask).convert('L')
    salida = _LAMA(img_pil, mask_pil).convert('RGB')
    return cv2.cvtColor(np.array(salida), cv2.COLOR_RGB2BGR)


def inpaint(img_bgr, mask):
    """Repara la región marcada. Si LaMa no está conectado, usa el método clásico
    de Telea como sustituto, que sirve para probar la interfaz de extremo a extremo
    pero NO es el modelo del trabajo."""
    if mask is None or not mask.any():
        return img_bgr, 'sin máscara'
    if LAMA_DISPONIBLE:
        return inpaint_lama(img_bgr, mask), 'LaMa'
    return cv2.inpaint(img_bgr, mask, 3, cv2.INPAINT_TELEA), 'Telea (provisional)'


try:
    _LAMA = cargar_lama()
    LAMA_DISPONIBLE = True
    print('LaMa listo en', DEVICE)
except Exception as e:
    print(f'LaMa no disponible ({type(e).__name__}: {e})')
    print('La interfaz seguirá con el método clásico como sustituto.')

## 3. Salvaguardas

Dos de las medidas de la sección de ética se implementan aquí.

El **aviso sobre rostros** compara la máscara con los rostros detectados en la imagen. No
pretende ser un detector fiable, sino levantar la mano cuando la reconstrucción cae sobre
una cara, que es donde una inferencia incorrecta tiene consecuencias sobre la identidad
de la persona retratada. Se emplea el clasificador en cascada que incluye OpenCV, sin
dependencias adicionales.

El **pie identificativo** deja constancia en la propia imagen de que la salida es una
versión procesada, distinta del documento original.

In [ ]:
_CASCADE = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

def solapa_con_rostro(img_bgr, mask, umbral=0.02):
    """Devuelve (hay_solape, n_rostros). Umbral: fracción del rostro cubierta."""
    if mask is None or not mask.any():
        return False, 0
    gris = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    caras = _CASCADE.detectMultiScale(gris, scaleFactor=1.1, minNeighbors=5, minSize=(40,40))
    for (x, y, w, h) in caras:
        region = mask[y:y+h, x:x+w]
        if region.size and (region > 0).mean() > umbral:
            return True, len(caras)
    return False, len(caras)


def pie_identificativo(img_bgr, texto='Versión restaurada mediante IA · no es el documento original'):
    """Añade una banda inferior con la advertencia. Sobre la imagen, no en metadatos,
    para que sobreviva a capturas de pantalla y reenvíos."""
    h, w = img_bgr.shape[:2]
    alto = max(22, int(h * 0.030))
    lienzo = np.full((h + alto, w, 3), 245, np.uint8)
    lienzo[:h] = img_bgr
    pil = Image.fromarray(cv2.cvtColor(lienzo, cv2.COLOR_BGR2RGB))
    d = ImageDraw.Draw(pil)
    try:
        fuente = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf',
                                    max(11, int(alto*0.52)))
    except Exception:
        fuente = ImageFont.load_default()
    d.text((int(w*0.012), h + alto*0.24), texto, fill=(90, 90, 90), font=fuente)
    return cv2.cvtColor(np.array(pil), cv2.COLOR_RGB2BGR)

## 4. Pipeline

In [ ]:
def _limitar(img_bgr, lado_max=LADO_MAX):
    h, w = img_bgr.shape[:2]
    if max(h, w) <= lado_max:
        return img_bgr, False
    f = lado_max / max(h, w)
    return cv2.resize(img_bgr, (int(round(w*f)), int(round(h*f))),
                      interpolation=cv2.INTER_AREA), True


def _mascara_del_editor(editor):
    """ImageEditor devuelve background, layers y composite. La máscara es el canal
    alfa de las capas pintadas por el usuario."""
    fondo = editor.get('background')
    capas = editor.get('layers') or []
    if fondo is None:
        return None, None
    fondo = np.asarray(fondo)[:, :, :3]
    mask = np.zeros(fondo.shape[:2], np.uint8)
    for capa in capas:
        c = np.asarray(capa)
        if c.ndim == 3 and c.shape[2] == 4:
            mask[c[..., 3] > 8] = 255
        elif c.ndim == 3:
            mask[c.max(axis=2) > 8] = 255
    return cv2.cvtColor(fondo, cv2.COLOR_RGB2BGR), mask


def restaurar(editor, modelo, marcar_salida, progress=gr.Progress()):
    if editor is None:
        raise gr.Error('Carga primero una fotografía.')

    img_bgr, mask = _mascara_del_editor(editor)
    if img_bgr is None:
        raise gr.Error('No se ha podido leer la imagen.')

    avisos = []
    img_bgr, reducida = _limitar(img_bgr)
    if reducida:
        mask = cv2.resize(mask, (img_bgr.shape[1], img_bgr.shape[0]),
                          interpolation=cv2.INTER_NEAREST)
        avisos.append(f'Entrada reducida a {LADO_MAX} px de lado mayor.')

    # Aviso ético ANTES de procesar
    hay_cara, n_caras = solapa_con_rostro(img_bgr, mask)
    if hay_cara:
        avisos.append('La zona marcada se solapa con un rostro detectado. '
                      'La reconstrucción facial procede del conocimiento previo del '
                      'modelo, no de la fotografía: trátala como de menor confianza.')

    t0 = time.time()
    progress(0.15, desc='Reparando la región marcada')
    reparada, metodo = inpaint(img_bgr, mask)

    progress(0.55, desc='Aumentando la resolución')
    if modelo not in UPS:
        raise gr.Error(f'Pesos no disponibles para «{modelo}».')
    with contextlib.redirect_stdout(io.StringIO()):
        sr, _ = UPS[modelo].enhance(reparada, outscale=ESCALA)

    progress(0.9, desc='Preparando la comparación')
    # El original se reescala solo para poder compararlo lado a lado. El deslizador
    # necesita ambas imágenes del mismo tamaño, así que el pie identificativo va
    # únicamente en la copia destinada a descargarse.
    ref = cv2.resize(img_bgr, (sr.shape[1], sr.shape[0]), interpolation=cv2.INTER_CUBIC)
    descarga = pie_identificativo(sr) if marcar_salida else sr

    px = int((mask > 0).sum()) if mask is not None else 0
    resumen = (f'**{time.time()-t0:.1f} s** · inpainting: {metodo} · '
               f'super-resolución: A-ESRGAN {modelo} ×{ESCALA}\n\n'
               f'{img_bgr.shape[1]}×{img_bgr.shape[0]} → {sr.shape[1]}×{sr.shape[0]} · '
               f'{px} píxeles marcados · {n_caras} rostro(s) detectado(s)')
    if avisos:
        resumen += '\n\n' + '\n\n'.join('⚠ ' + a for a in avisos)

    return ((cv2.cvtColor(ref, cv2.COLOR_BGR2RGB),
             cv2.cvtColor(sr, cv2.COLOR_BGR2RGB)),
            cv2.cvtColor(descarga, cv2.COLOR_BGR2RGB),
            resumen)

## 5. Interfaz

In [ ]:
DESCRIPCION = """
Carga una fotografía, pinta encima las zonas dañadas y ejecuta la restauración.
El pipeline repara la región marcada y multiplica por cuatro la resolución.

**No colorea ni estiliza**: solo repara daño y aumenta resolución. La comparación se
muestra siempre junto al original, para que el resultado pueda contrastarse con la
imagen de partida en lugar de recibirse como si fuera la fotografía original.
"""

with gr.Blocks(title='Restauración de fotografía histórica') as demo:
    gr.Markdown('# Restauración y super-resolución de fotografía histórica')
    gr.Markdown(DESCRIPCION)

    with gr.Row():
        with gr.Column(scale=1):
            editor = gr.ImageEditor(
                label='Fotografía · pinta las zonas dañadas',
                type='numpy', image_mode='RGB',
                sources=('upload', 'clipboard'),
                layers=False, transforms=(),
                brush=gr.Brush(colors=['#FF3B30'], color_mode='fixed', default_size=18),
                height=430)
            modelo = gr.Radio(list(UPS) or ['preentrenado'],
                              value=('ajustado' if 'ajustado' in UPS else 'preentrenado'),
                              label='Modelo de super-resolución',
                              info='Comparar el preentrenado con el ajustado en el dominio')
            marcar = gr.Checkbox(value=True, label='Añadir pie identificativo a la salida',
                                 info='Deja constancia de que la imagen ha sido procesada')
            boton = gr.Button('Restaurar', variant='primary')

        with gr.Column(scale=1):
            comparador = gr.ImageSlider(label='Original / restaurada', type='numpy',
                                        height=430)
            info = gr.Markdown()
            # buttons sustituye a show_download_button desde Gradio 6.0
            descarga = gr.Image(label='Imagen para descargar', type='numpy',
                                height=200, buttons=['download', 'fullscreen'])

    gr.Markdown(
        '---\n*Las imágenes restauradas son versiones procesadas mediante modelos '
        'generativos y no sustituyen al documento original. En las regiones reparadas, '
        'el contenido procede de la distribución aprendida por el modelo y no de la '
        'fotografía de partida.*')

    boton.click(restaurar, inputs=[editor, modelo, marcar],
                outputs=[comparador, descarga, info])

print('interfaz construida')

In [ ]:
# theme se pasa en launch() desde Gradio 6.0, no en el constructor de Blocks
demo.queue(max_size=8).launch(share=True, debug=False, theme=gr.themes.Soft())

---

### Qué falta para el despliegue permanente

Conectar LaMa en el adaptador de la sección 2, que es lo único imprescindible. Mientras
tanto la interfaz funciona con el método clásico de Telea como sustituto, lo que permite
validar el flujo completo pero **no** corresponde al modelo del trabajo.

Después, el mismo código se sube a un Space de Hugging Face. Los cambios son tres: mover
el contenido de las celdas a un `app.py`, declarar las dependencias en `requirements.txt`
fijando una versión de `torchvision` compatible con `basicsr` para no depender del parche,
y decorar la función de inferencia con `@spaces.GPU` para que ZeroGPU asigne la tarjeta
durante la ejecución.